# 05. Blueprints & Extension Points

Two ideas ship together in SPEC-047, and they solve two different problems.

**Extension points** are *how arcagent stays a small, closed core while still being pluggable*. Rather than every optional subsystem (memory, skill self-improvement, ...) hand-rolling its own "pick an implementation from config" dispatch, choice-gate, and BYO-import logic, arcagent declares exactly two extension *shapes* — **select-one** (pick one implementation by config setting: `brain`, `skills`) and **scan-many** (a read-only view over already-loaded capabilities: `tools`, `hook-builds`) — and every seam is a thin declarative instance of one of them. The payoff: one audited, fail-closed dynamic-import gate protects every pluggable subsystem, not four different ones with four different bug surfaces.

**Blueprints** are *how a deployment gets bootstrapped in one command* instead of hand-editing `arcagent.toml`. A blueprint is a signed, versioned TOML preset — `arc init --blueprint federal-analyst` or `arc blueprint apply enterprise-ops` — that's deep-merged UNDER your own config (you always win) and can only *raise* the security tier floor, never lower it.

Both ideas share one thread: **a deployment operator should be able to see, in one command, exactly what's plugged into an agent and whether it's safe to trust — and nothing should load unverified code to answer that question.** That's `arc ext inspect` / `arc ext verify`, the closing section of this notebook.


## 1. Setup

The setup cell below is the standard Arc walkthrough preamble: it locates the repo root, adds every package's `src/` (and `tests/` where present) to `sys.path`, and loads any `.env` file at the root. Identical across every notebook.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")
print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: /Users/joshschultz/Projects/arc


Quick smoke test — confirm the SPEC-047 modules import cleanly.

In [2]:
import arcagent
from arcagent.extension import ExtensionPoint, select_extension
from arcagent.extension.families import FAMILIES, ScanManyFamily, SelectOneFamily
from arcagent.extension.inspect import ExtensionStatus, inspect_extensions
from arcagent.blueprints import (
    ResolvedBlueprint,
    apply_blueprint,
    dumps_toml,
    list_blueprints,
    resolve_blueprint,
)
from arcagent.tiers import RELAXABLE_KNOBS, RelaxableKnob, resolve_tier_floor, stricter_tier, tier_rank

print(f"arcagent version: {arcagent.__version__}")
print("Loaded extension + blueprint + tiers surface OK.")


arcagent version: 0.15.0
Loaded extension + blueprint + tiers surface OK.


## 2. The four extension-point families

Arc has exactly two extension *shapes*. The four roadmap-named families each map onto one:

- **select-one** (`brain`, `skills`) — one `ExtensionPoint` picked by a single config setting.
- **scan-many** (`tools`, `hook-builds`) — a read-only VIEW over the existing SPEC-021 `CapabilityRegistry`, filtered by decorator kind. No new loader, inspection only.

`FAMILIES` is the literal declared tuple — the single source of truth `arc ext inspect` iterates over.

In [3]:
for family in FAMILIES:
    if isinstance(family, SelectOneFamily):
        print(f"{family.name:12s} select_one   setting_path={family.setting_path}  allowlist_key={family.allowlist_key!r}")
    else:
        print(f"{family.name:12s} scan_many    kinds={sorted(family.kinds)}")


brain        select_one   setting_path=('memory', 'brain')  allowlist_key='brain_allowlist'
skills       select_one   setting_path=('skills', 'adapter')  allowlist_key='adapter_allowlist'
tools        scan_many    kinds=['tool']
hook-builds  scan_many    kinds=['background_task', 'capability', 'hook']


A fifth registry kind exists (`"skill"` — loaded `SKILL.md` capability folders) but it's deliberately **not** a scan-many family here: the `skills` *select-one* family is the improver **adapter** (SPEC-044's `SkillAdapter`), a different concept from loaded skill folders (those surface via `arc skill`, not `arc ext`).

## 3. The `ExtensionPoint` descriptor — one mechanism, declared not duplicated

Before SPEC-047, `brain/select.py` (SPEC-041) and `skilladapt/select.py` (SPEC-044) each hand-rolled the same three things: dispatch on a config string, a fail-closed BYO allowlist gate, and a dotted-path importer. SPEC-047 pulls that mechanism out once into `arcagent.extension.select.select_extension`, and each seam now just *declares* the four things that actually differ between them as a frozen `ExtensionPoint`:

| Field | Meaning |
|---|---|
| `null_factory` | zero-arg callable returning the Null default (`none`/`""`/`null`, or a degraded builtin) |
| `builtin_modules` | choice string → the real import string (e.g. `{"arcskill": "arcskill.improver"}`) |
| `builtin_builder` | `(imported_module, context) -> instance \| None` |
| `byo_constructor` | `(resolved_cls, context) -> instance` — how a BYO class gets built |

Here's the real `skills` point, straight from `arcagent/skilladapt/select.py`:
```python
_SKILLADAPT_POINT = ExtensionPoint(
    name="skills",
    null_factory=NullSkillAdapter,
    builtin_modules={"arcskill": "arcskill.improver"},
    builtin_builder=_build_arcskill,
    byo_constructor=lambda cls, ctx: cls(ctx["workspace"]),
)
```
`arcagent/brain/select.py` declares `_BRAIN_POINT` the same way for SPEC-041's `Brain` seam — same shape, different builtin (`arcmemory`) and BYO constructor arity. A third extension point would cost one more `ExtensionPoint(...)` declaration, not a fourth copy of the mechanism.

In [4]:
from arcagent.skilladapt.select import _SKILLADAPT_POINT

print("name           :", _SKILLADAPT_POINT.name)
print("kind           :", _SKILLADAPT_POINT.kind)
print("null_factory   :", _SKILLADAPT_POINT.null_factory)
print("builtin_modules:", dict(_SKILLADAPT_POINT.builtin_modules))


name           : skills
kind           : select_one
null_factory   : <class 'arcagent.skilladapt.protocol.NullSkillAdapter'>
builtin_modules: {'arcskill': 'arcskill.improver'}


### The shared dispatch: `select_extension`

`select_extension(point, setting, *, tier, allowlist, context, logger)` is the ONE copy of: `none`/`""`/`null` → the Null default; a builtin choice → lazy import + build (silently degrading for `auto`, warning otherwise); anything else → a dotted `module.Class` BYO path, REFUSED before any import above the personal tier unless it's on the operator allowlist (importing an unverified dotted path at startup is arbitrary code execution — ASI04). Let's drive it directly against the real `skills` point.

In [5]:
import logging

_logger = logging.getLogger("walkthrough.extension")

# 1. "none" -> the Null default, no import at all.
none_adapter = select_extension(
    _SKILLADAPT_POINT,
    "none",
    tier="personal",
    allowlist=(),
    context={},
    logger=_logger,
)
print("setting='none'    ->", type(none_adapter).__name__)

# 2. a builtin choice ("arcskill") -> lazy import + build via _build_arcskill.
arcskill_context = {
    "workspace": Path("/tmp/walkthrough-skills-workspace"),
    "config": {},
    "tier": "personal",
    "llm": None,
    "signer": None,
    "approval_provider": None,
    "eval_runner": None,
    "audit_sink": None,
    "agent_did": "",
    "skill_path": None,
}
arcskill_adapter = select_extension(
    _SKILLADAPT_POINT,
    "arcskill",
    tier="personal",
    allowlist=(),
    context=arcskill_context,
    logger=_logger,
)
print("setting='arcskill' ->", type(arcskill_adapter).__name__)


setting='none'    -> NullSkillAdapter


setting='arcskill' ->

 ArcSkillImprover


### The fail-closed BYO gate

A dotted class path above the personal tier that is NOT on the operator allowlist is refused *before any import happens* — the exact ASI04 (agentic supply chain) mitigation this whole mechanism exists for.

In [6]:
try:
    select_extension(
        _SKILLADAPT_POINT,
        "some.untrusted.module:EvilAdapter",
        tier="enterprise",  # above personal
        allowlist=(),  # NOT allowlisted
        context=arcskill_context,
        logger=_logger,
    )
    print("UNEXPECTED: import was not refused")
except ValueError as exc:
    print("Refused fail-closed, as expected:")
    print(" ", exc)


Refused fail-closed, as expected:
  BYO skills class-path 'some.untrusted.module:EvilAdapter' is not on the operator allowlist; refusing to import an unverified class-path at tier 'enterprise' (fail-closed)


In [7]:
# The personal tier is more permissive: an unallowlisted BYO path is accepted at personal
# (it just won't successfully import a module that doesn't exist -- ImportError, not a
# security refusal). Demonstrate the allowlist gate is tier-scoped, not import-scoped.
try:
    select_extension(
        _SKILLADAPT_POINT,
        "some.untrusted.module:EvilAdapter",
        tier="personal",
        allowlist=(),
        context=arcskill_context,
        logger=_logger,
    )
except ValueError as exc:
    print("Refused at personal too (unexpected):", exc)
except ModuleNotFoundError as exc:
    print("At personal tier the allowlist gate does NOT fire -- it proceeds straight to import,")
    print("which then fails for the ordinary reason the module doesn't exist:")
    print(" ", exc)


At personal tier the allowlist gate does NOT fire -- it proceeds straight to import,
which then fails for the ordinary reason the module doesn't exist:
  No module named 'some'


## 4. `arc ext inspect` / `arc ext verify` — pure-read inspection

`inspect_extensions(config, registry=None, *, trusted_public_key=None)` reports, per family, what's selected, whether it's available, and its signed status — **without ever importing a BYO module to check it** (a select-one builtin's availability is probed with `importlib.util.find_spec`, which locates a module without executing it; a BYO dotted path is judged purely by the allowlist gate). This is exactly the data `arc ext inspect` renders and `arc ext verify` gates a CI pipeline on.

Let's build a minimal real `ArcAgentConfig` (the same flat-read shape the agent runtime and `arc ext` both use) and inspect it.

In [8]:
from arcagent.core.config import ArcAgentConfig

raw_config = {
    "agent": {"name": "walkthrough-agent"},
    "llm": {"model": "none"},
    "security": {"tier": "personal"},
    "modules": {
        "memory": {"enabled": True, "config": {"brain": "none"}},
        "skills": {"enabled": True, "config": {"adapter": "arcskill"}},
    },
}
config = ArcAgentConfig.model_validate(raw_config)
print("tier:", config.security.tier)
print("modules.skills.config:", config.modules["skills"].config)


tier: personal
modules.skills.config: {'adapter': 'arcskill'}


In [9]:
rows = inspect_extensions(config, registry=None)
print(f"{'family':10s} {'kind':10s} {'selected':10s} {'available':10s} signed")
for r in rows:
    print(f"{r.family:10s} {r.kind:10s} {r.selected:10s} {str(r.available):10s} {r.signed}")


family     kind       selected   available  signed
brain      select_one none       True       n/a
skills     select_one arcskill   True       builtin


Only the two select-one families show up because we passed `registry=None` — scan-many rows require a live `CapabilityRegistry` (built by the arcagent runtime at startup). Now flip the `skills` adapter to an unallowlisted BYO path above personal and watch the row read `refused`.

In [10]:
raw_config_refused = dict(raw_config)
raw_config_refused["security"] = {"tier": "enterprise"}
raw_config_refused["modules"] = {
    **raw_config["modules"],
    "skills": {"enabled": True, "config": {"adapter": "some.rogue.module:Adapter"}},
}
config_refused = ArcAgentConfig.model_validate(raw_config_refused)
rows_refused = inspect_extensions(config_refused, registry=None)
for r in rows_refused:
    if r.family == "skills":
        print(r)


ExtensionStatus(family='skills', kind='select_one', selected='some.rogue.module:Adapter', available=False, signed='refused', detail='')


### The CLI surface

`arc ext inspect --agent DIR` renders exactly the table above. `arc ext verify --agent DIR` reuses the same `inspect_extensions()` call, then applies one rule — `_would_refuse(row, tier)`: a select-one row is a refusal if `signed == "refused"` or `not available`; a scan-many row is a refusal if it's `unsigned` at enterprise/federal. It exits non-zero on any refusal, so it's usable as a federal change-control CI gate. Let's drive the real console script end-to-end against a scratch agent directory.

In [11]:
import subprocess
import sys
import tempfile

# The `arc` console script lives beside the interpreter running this kernel
# (installed into the venv's bin/ dir) -- resolve it relative to sys.executable
# rather than relying on it being on the notebook process's PATH.
ARC_BIN = str(Path(sys.executable).parent / "arc")

with tempfile.TemporaryDirectory() as scratch:
    scratch_path = Path(scratch)
    (scratch_path / "arcagent.toml").write_text(
        dumps_toml(
            {
                "agent": {"name": "cli-demo"},
                "llm": {"model": "none"},
                "security": {"tier": "personal"},
                "modules": {"skills": {"enabled": True, "config": {"adapter": "none"}}},
            }
        ),
        encoding="utf-8",
    )
    result = subprocess.run(
        [ARC_BIN, "ext", "inspect", "--agent", str(scratch_path)],
        capture_output=True,
        text=True,
    )
    print("exit code:", result.returncode)
    print(result.stdout)
    if result.stderr:
        print("stderr:", result.stderr)


exit code: 0
Family  Kind        Selected      Available  Signed  
------  ----------  ------------  ---------  --------
brain   select_one  none          yes        n/a     
skills  select_one  none          yes        n/a     
tools   scan_many   bash          yes        unsigned
tools   scan_many   create_skill  yes        unsigned
tools   scan_many   create_tool   yes        unsigned
tools   scan_many   edit          yes        unsigned
tools   scan_many   find          yes        unsigned
tools   scan_many   grep          yes        unsigned
tools   scan_many   ls            yes        unsigned
tools   scan_many   read          yes        unsigned
tools   scan_many   reload        yes        unsigned
tools   scan_many   update_skill  yes        unsigned
tools   scan_many   update_tool   yes        unsigned
tools   scan_many   write         yes        unsigned



In [12]:
with tempfile.TemporaryDirectory() as scratch:
    scratch_path = Path(scratch)
    (scratch_path / "arcagent.toml").write_text(
        dumps_toml(
            {
                "agent": {"name": "cli-demo-verify"},
                "llm": {"model": "none"},
                "security": {"tier": "personal"},
                "modules": {"skills": {"enabled": True, "config": {"adapter": "none"}}},
            }
        ),
        encoding="utf-8",
    )
    result = subprocess.run(
        [ARC_BIN, "ext", "verify", "--agent", str(scratch_path)],
        capture_output=True,
        text=True,
    )
    print("exit code:", result.returncode, "(0 == nothing would be refused at load)")
    print(result.stdout)


exit code: 0 (0 == nothing would be refused at load)
All extension-point selections load-clean at tier 'personal'.



## 5. Signed blueprints — bootstrap a deployment in one command

A blueprint is a `[blueprint]`-headed TOML carrying a config overlay of the same shape as `arcagent.toml`. Three things matter:

1. **Materialize-to-disk, not a runtime layer.** The agent runtime (`arcagent/__main__.py`) flat-reads its `arcagent.toml` — there's no merge hook at boot. `arc init --blueprint NAME` / `arc blueprint apply NAME` render the preset under your values and **write the concrete file**.
2. **Precedence: packaged-defaults < blueprint < user.** A blueprint is a starting point — your explicit keys always win. It's a write-time deep-merge, never a clobber overwrite (unlike `arc agent build`).
3. **Tier floor: `effective_tier = stringency-max(deployment, blueprint)`.** A blueprint can only *raise* the security tier, never lower it — and the real `SecurityConfig` validator then forces every federal floor (FIPS, vault_transit custody, ...) at load, so "a personal blueprint can't weaken federal" is true by construction.

Let's see what ships packaged inside the wheel.

In [13]:
packaged = list_blueprints()
for bp in packaged:
    print(f"{bp.name:20s} v{bp.version:6s} tier={bp.tier:10s} source={bp.source:9s} signed={bp.signed}")


enterprise-ops       v1.0.0  tier=enterprise source=packaged  signed=True
federal-analyst      v1.0.0  tier=federal    source=packaged  signed=True
personal-assistant   v1.0.0  tier=personal   source=packaged  signed=True


In [14]:
demo_bp = resolve_blueprint("enterprise-ops", tier="personal")
print("name       :", demo_bp.name)
print("version    :", demo_bp.version)
print("tier       :", demo_bp.tier)
print("source     :", demo_bp.source)
print("signed     :", demo_bp.signed, "(packaged presets are provenance-trusted, no .arcsig needed)")
print("sha256[:16]:", demo_bp.sha256[:16])
print("overlay    :")
for k, v in demo_bp.overlay.items():
    print(f"  {k}: {v}")


name       : enterprise-ops
version    : 1.0.0
tier       : enterprise
source     : packaged
signed     : True (packaged presets are provenance-trusted, no .arcsig needed)
sha256[:16]: 7ccc43f88cf3dfcc
overlay    :
  security: {'tier': 'enterprise', 'custody': 'vault_transit'}
  telemetry: {'enabled': True, 'export_traces': True}
  modules: {'memory': {'enabled': True, 'config': {'brain': 'arcmemory', 'tier': 'enterprise'}}, 'skills': {'enabled': True, 'config': {'adapter': 'arcskill', 'tier': 'enterprise'}}}


### Deep-merge + tier floor in action

`apply_blueprint(blueprint, base, *, deployment_tier)` deep-merges the blueprint's overlay UNDER `base` (your existing values win) and sets `[security].tier` to the stringency-max of the deployment tier, the blueprint's own tier, and whatever the merge result ends up with.

In [15]:
# Deployment starts with nothing but a personal-tier default. The enterprise-ops blueprint
# raises the floor to "enterprise" and turns on skills/memory.
base = {"agent": {"name": "my-agent"}, "llm": {"model": "none"}}
merged = apply_blueprint(demo_bp, base, deployment_tier="personal")
print("effective tier         :", merged["security"]["tier"])
print("modules.skills.config  :", merged["modules"]["skills"]["config"])
print("agent.name (user value, preserved):", merged["agent"]["name"])


effective tier         : enterprise
modules.skills.config  : {'adapter': 'arcskill', 'tier': 'enterprise'}
agent.name (user value, preserved): my-agent


In [16]:
# Now show the floor cannot be LOWERED: the deployment is already federal; applying the
# personal-assistant blueprint (tier="personal") must not weaken it.
personal_bp = resolve_blueprint("personal-assistant", tier="personal")
merged_from_federal = apply_blueprint(personal_bp, base, deployment_tier="federal")
print("deployment_tier=federal + blueprint.tier=personal -> effective tier:", merged_from_federal["security"]["tier"])
assert merged_from_federal["security"]["tier"] == "federal", "a blueprint must never lower the floor"
print("Confirmed: the blueprint could not weaken the federal floor.")


deployment_tier=federal + blueprint.tier=personal -> effective tier: federal
Confirmed: the blueprint could not weaken the federal floor.


### Signed user blueprints, pinned to the operator key

Packaged presets ship inside the verified wheel (provenance-trusted). A **user** preset from `~/.arc/blueprints/` is different: above the personal tier it MUST be verified via its `.arcsig` sidecar, PINNED to the deployment operator's public key — the same key `arc blueprint sign` signs with. An unpinned signature check would let an attacker self-sign a malicious preset with a throwaway keypair and have it verify "successfully". Let's build one for real, sign it, and show both the success path and the fail-closed refusal.

In [17]:
import tempfile

from arcagent.capabilities.artifact_signing import write_signature
from arctrust import generate_keypair

operator_keypair = generate_keypair()
attacker_keypair = generate_keypair()

user_blueprint_toml = '''\
[blueprint]
name = "walkthrough-custom"
version = "1.0.0"
tier = "enterprise"
description = "A hand-authored user blueprint for this walkthrough."

[telemetry]
enabled = true
'''

user_dir = Path(tempfile.mkdtemp(prefix="arc-blueprints-"))
bp_path = user_dir / "walkthrough-custom.toml"
bp_path.write_text(user_blueprint_toml, encoding="utf-8")

sidecar = write_signature(
    bp_path,
    bp_path.read_bytes(),
    signer_did=f"operator:{operator_keypair.public_key.hex()[:16]}",
    private_key=operator_keypair.private_key,
)
print("wrote preset :", bp_path)
print("wrote sidecar:", sidecar)


wrote preset : /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc-blueprints-udmphxvb/walkthrough-custom.toml
wrote sidecar: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc-blueprints-udmphxvb/walkthrough-custom.toml.arcsig


In [18]:
# Verified: resolved with the SAME operator public key it was signed with, above personal tier.
resolved_ok = resolve_blueprint(
    "walkthrough-custom",
    tier="enterprise",
    user_dir=user_dir,
    operator_public_key=operator_keypair.public_key,
)
print("signed :", resolved_ok.signed)
print("tier   :", resolved_ok.tier)
print("signer :", resolved_ok.signer_did)


signed : True
tier   : enterprise
signer : operator:0a8fe37bfae741ec


In [19]:
# Refused: pinned against the WRONG public key (an attacker who self-signed with their own
# keypair) above the personal tier -- fail-closed, not silently "unsigned".
try:
    resolve_blueprint(
        "walkthrough-custom",
        tier="enterprise",
        user_dir=user_dir,
        operator_public_key=attacker_keypair.public_key,
    )
    print("UNEXPECTED: a wrong-key preset was accepted")
except ValueError as exc:
    print("Refused fail-closed, as expected:")
    print(" ", exc)


Refused fail-closed, as expected:
  user blueprint 'walkthrough-custom' is unsigned or not signed by the deployment operator key; refusing to apply above the personal tier (fail-closed, LLM03/ASI04)


In [20]:
# Refused: no operator key resolvable at all above personal -- an unpinned floor is no floor.
try:
    resolve_blueprint(
        "walkthrough-custom",
        tier="enterprise",
        user_dir=user_dir,
        operator_public_key=None,
    )
    print("UNEXPECTED: resolution succeeded with no pinned key")
except ValueError as exc:
    print("Refused fail-closed (no operator key to pin against):")
    print(" ", exc)


Refused fail-closed (no operator key to pin against):
  user blueprint 'walkthrough-custom' requires the deployment operator's public key to pin its signature against, but it could not be resolved; refusing above the personal tier (fail-closed — an unpinned signature gate accepts any self-signed preset, LLM03/ASI04)


In [21]:
# At the personal tier, an unsigned/unpinned preset is allowed with only an audit-warn --
# personal deployments favor zero-friction bootstrap over verified provenance.
resolved_personal = resolve_blueprint(
    "walkthrough-custom",
    tier="personal",
    user_dir=user_dir,
    operator_public_key=None,
)
print("resolved at personal tier, signed=", resolved_personal.signed, "(audit-warn only, not refused)")


resolved at personal tier, signed= True (audit-warn only, not refused)


### Mapping to the CLI

| Verb | Does |
|---|---|
| `arc blueprint list` | packaged + `~/.arc/blueprints` presets (name/version/tier/signed) |
| `arc blueprint show NAME` | print the resolved overlay |
| `arc blueprint apply NAME [--agent DIR] [--dry-run]` | verify → deep-merge UNDER the target's config → write |
| `arc blueprint verify NAME` | signature validity for the current tier |
| `arc blueprint sign PATH` | operator-sign a user blueprint (writes the `.arcsig` sidecar) |
| `arc init --blueprint NAME` | bootstrap a brand-new agent directory from a preset |

`apply`/`init --blueprint` route their audit trail through one shared callback (`arccli.commands.blueprint.audit_apply`): every granted tier relaxation (`tier.relaxation_granted`, one per relaxed knob — see §6 below) plus a final `blueprint.applied` event, both going to the operator WORM sink at enterprise/federal or a structured log at personal. Let's drive the real CLI end-to-end.

In [22]:
import subprocess

with tempfile.TemporaryDirectory() as scratch:
    scratch_path = Path(scratch)
    result = subprocess.run(
        [ARC_BIN, "blueprint", "list"],
        capture_output=True,
        text=True,
        env={**os.environ, "HOME": str(scratch_path)},
    )
    print("exit code:", result.returncode)
    print(result.stdout)


exit code: 0
Name                Version  Tier        Source    Signed    
------------------  -------  ----------  --------  ----------
enterprise-ops      1.0.0    enterprise  packaged  provenance
federal-analyst     1.0.0    federal     packaged  provenance
personal-assistant  1.0.0    personal    packaged  provenance



In [23]:
with tempfile.TemporaryDirectory() as scratch:
    scratch_path = Path(scratch)
    result = subprocess.run(
        [ARC_BIN, "blueprint", "show", "personal-assistant"],
        capture_output=True,
        text=True,
        env={**os.environ, "HOME": str(scratch_path)},
    )
    print("exit code:", result.returncode)
    print(result.stdout)


exit code: 0
# blueprint: personal-assistant v1.0.0 (tier=personal, source=packaged)
[security]
tier = "personal"

[modules]

[modules.memory]
enabled = true

[modules.memory.config]
brain = "arcmemory"

[modules.skills]
enabled = true

[modules.skills.config]
adapter = "arcskill"



## 6. Config-relaxable tiers — `RelaxableKnob`

Before SPEC-047, five different modules hand-rolled the same "personal can relax this / federal pins it" idiom. `arcagent.tiers` declares it once: `RELAXABLE_KNOBS` is a table of every tier-relaxable security knob, its federal floor, and whether personal/enterprise may loosen it below that floor. `resolve_tier_floor(knob, tier, requested, *, was_set, audit=None)` is the one shared decision:

- **federal** forces the floor and REJECTS any explicit weaker value outright.
- **personal / enterprise** may relax the knob *if the knob permits it* — and a granted relaxation is audited (`tier.relaxation_granted`).

In [24]:
print(f"{'knob':22s} {'federal_floor':16s} {'relax_personal':16s} {'relax_enterprise':18s} {'stricter_is':12s} enforcement")
for knob in RELAXABLE_KNOBS:
    print(
        f"{knob.name:22s} {str(knob.federal_floor):16s} {str(knob.relax_personal):16s} "
        f"{str(knob.relax_enterprise):18s} {knob.stricter_is:12s} {knob.enforcement}"
    )


knob                   federal_floor    relax_personal   relax_enterprise   stricter_is  enforcement
require_fips           True             True             True               exact        SecurityConfig
custody                vault_transit    True             True               exact        SecurityConfig
signing_algorithm      ecdsa-p256       True             True               exact        SecurityConfig
runaway_max_repeat     8                True             True               smaller      SecurityConfig
error_cascade_max      5                True             True               smaller      SecurityConfig
budget.max_tokens      500000           True             True               smaller      dispatch
budget.max_cost_usd    10.0             True             True               smaller      dispatch
budget.max_requests    500              True             True               smaller      dispatch
allow_all_imports      False            True             True               exact    

In [25]:
require_fips_knob = next(k for k in RELAXABLE_KNOBS if k.name == "require_fips")

# Federal REJECTS an explicit weaker value outright.
try:
    resolve_tier_floor(require_fips_knob, "federal", False, was_set=True)
    print("UNEXPECTED: federal accepted require_fips=False")
except ValueError as exc:
    print("federal + require_fips=False (explicit) -> rejected:")
    print(" ", exc)


federal + require_fips=False (explicit) -> rejected:
  federal tier requires require_fips=True (SC-5/SC-13/IA-7) — refusing a looser/disabled value


In [26]:
# Federal, unset -> silently pinned to the floor (no error, no explicit weaker value given).
pinned = resolve_tier_floor(require_fips_knob, "federal", None, was_set=False)
print("federal + require_fips unset -> pinned to:", pinned)

# Personal MAY relax it, and a granted relaxation is audited.
captured_audit_events = []
relaxed = resolve_tier_floor(
    require_fips_knob,
    "personal",
    False,
    was_set=True,
    audit=lambda event, detail: captured_audit_events.append((event, detail)),
)
print("personal + require_fips=False (explicit) -> allowed:", relaxed)
print("audited event:", captured_audit_events[0])


federal + require_fips unset -> pinned to: True
personal + require_fips=False (explicit) -> allowed: False
audited event: ('tier.relaxation_granted', {'knob': 'require_fips', 'tier': 'personal', 'requested': False, 'resolved': False})


In [27]:
# A "smaller-is-stricter" knob behaves in the opposite comparison direction: a SMALLER
# number is MORE strict, so a LARGER requested value is what gets rejected at federal.
runaway_knob = next(k for k in RELAXABLE_KNOBS if k.name == "runaway_max_repeat")
print("runaway_max_repeat federal floor:", runaway_knob.federal_floor, "(smaller = stricter)")

try:
    resolve_tier_floor(runaway_knob, "federal", 50, was_set=True)  # 50 > floor of 8: weaker
    print("UNEXPECTED: federal accepted a looser runaway_max_repeat")
except ValueError as exc:
    print("federal + runaway_max_repeat=50 (looser than the floor of 8) -> rejected:")
    print(" ", exc)

# A STRICTER-than-floor value (smaller number) is honored, not rejected, even at federal.
stricter_value = resolve_tier_floor(runaway_knob, "federal", 3, was_set=True)
print("federal + runaway_max_repeat=3 (stricter than the floor of 8) -> honored:", stricter_value)


runaway_max_repeat federal floor: 8 (smaller = stricter)
federal + runaway_max_repeat=50 (looser than the floor of 8) -> rejected:
  federal tier requires runaway_max_repeat=8 (SC-5/SC-13/IA-7) — refusing a looser/disabled value
federal + runaway_max_repeat=3 (stricter than the floor of 8) -> honored: 3


`enforcement="SecurityConfig"` knobs (the first five) are enforced by the delegating validator in `core/config.py` — this is what actually runs when you load an `ArcAgentConfig`. The remaining rows (`budget.*`, `allow_all_imports`) are declarative reference rows: `arc ext verify` and the audit trail read them, but enforcement lives at their own existing sites (the dispatch budget resolver, the dynamic-loader import policy) — `tiers.py` owns the *policy table*, not a second enforcement path.

## 7. Recap

- **Two extension shapes, four families.** `select-one` (`brain`, `skills`) and `scan-many` (`tools`, `hook-builds`) — declared once in `FAMILIES`, dispatched once by `select_extension`.
- **One `ExtensionPoint` per seam**, declaring only what differs: the Null default, the builtin import map, the builder, the BYO constructor.
- **`arc ext inspect` / `arc ext verify`** give an operator one pure-read command to see (and CI-gate on) exactly what's plugged in and whether it would load clean at the configured tier — never importing a BYO module just to check it.
- **Blueprints** bootstrap a deployment from a signed, versioned TOML preset in one command, always deep-merged UNDER your own values, and can only raise the tier floor — enforced by construction via `RelaxableKnob`/`resolve_tier_floor`, the single shared "personal-relaxable / federal-pinned" decision every tier-sensitive knob in Arc now goes through.

Tier vocabulary throughout Arc is exactly `federal > enterprise > personal` — there is no "open" tier.